# ViT Sprint 4.5 — GMU fusion ablation (Colab A100)

Runs the **GMU fusion** ablation (faithful element-wise gate, VLD-19) on the
multi-backbone sets — a **fold-0 screen** of 4 configs (3 pairs + triple) via
`scripts/train.py`, using the Sprint 3.5 **tuned config** (VLD-17). ~4 runs
(~30 min). A competitive config is promoted to 5-fold CV later (Slice 4).

**Dataset on Drive:** upload the dataset as one archive (`labeled-images.zip`/`.tar`
with `labeled-images/` at its root) under
`MyDrive/hyperkvasir-multi-cnn-fusion/data/hyperkvasir/` (cell 5; KI-VIT-001).

**Gates (VLD-11):** the env cell hard-asserts an A100; `scripts/train.py` aborts
off-A100. On Colab/Linux the tuned `num_workers` is safe (KI-VIT-002 is
Windows-only).

**Overnight-safe:** the training cell restores finished runs from Drive, skips any
run that already has `metrics.json`, streams progress, and backs each up to Drive.

**GMU screen ids** (multi-backbone only — GMU needs ≥2 branches, no singles):
`pair_vit_b_swin_t_gmu_cv`, `pair_vit_b_beit_b_gmu_cv`,
`pair_swin_t_beit_b_gmu_cv`, `triple_vit_swin_beit_gmu_cv`. Headline = triple-GMU
vs the Sprint 4 `11_triple_weighted_cv` fold-0 (0.6102). Do NOT start Sprint 5.

Prerequisite: the `sprint4.5/vit-gmu` branch is pushed to GitHub.

In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Clone/checkout the working branch and record the commit.
import os, subprocess
REPO_URL = 'https://github.com/YasinEkici/hyperkvasir-multi-backbone-fusion.git'
BRANCH = 'sprint4.5/vit-gmu'   # set to the branch that holds the code you want
REPO_DIR = '/content/hyperkvasir-multi-backbone-fusion'
# Public repo: no token needed. For a private repo, set
# os.environ['GITHUB_TOKEN'] = '...' in a scratch cell BEFORE running this one.
token = os.environ.get('GITHUB_TOKEN', '')
clone_url = REPO_URL.replace('https://', f'https://x-access-token:{token}@') if token else REPO_URL
if os.path.isdir(os.path.join(REPO_DIR, '.git')):
    os.chdir(REPO_DIR)
    # Checkout via FETCH_HEAD so this works even when the repo was first cloned
    # with --single-branch (origin/<BRANCH> tracking ref may not exist). The
    # gitignored data/ tree is preserved across the branch switch.
    subprocess.run(['git', 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', 'checkout', '-B', BRANCH, 'FETCH_HEAD'], check=True)
elif os.path.exists(REPO_DIR):
    raise RuntimeError(f'Path exists but is not a git repo: {REPO_DIR}. Restart the runtime.')
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', clone_url, REPO_DIR], check=True)
os.chdir(REPO_DIR)
!git rev-parse HEAD

In [ ]:
# 3. Build the isolated Colab env and hard-assert an A100 (VLD-11)
import os
REPO_DIR = '/content/hyperkvasir-multi-backbone-fusion'
os.chdir(REPO_DIR)
required_files = [
    'pyproject.toml',
    'env/requirements-colab.txt',
    'configs/vit/training/vit_finetune.yaml',
    'configs/vit/experiment_matrix.yaml',
]
missing = [path for path in required_files if not os.path.exists(path)]
if missing:
    !git branch --show-current
    !git rev-parse HEAD
    raise FileNotFoundError(f'Missing required repo files: {missing}. Push Slices 1-3 to GitHub and rerun cell 2.')
!python -m pip install -q uv
!uv venv --python 3.11 .venv
# Use `uv pip install -r` (resolves transitive deps); never `uv sync` here
# (local pyproject is pinned to CUDA 13.2 for the RTX 5080).
!uv pip install --python .venv/bin/python -r env/requirements-colab.txt
!uv run --no-sync python -c "import torch, timm; print('torch', torch.__version__, 'timm', timm.__version__); assert torch.cuda.is_available(), 'CUDA unavailable'; name=torch.cuda.get_device_name(0); print('device', name); assert 'A100' in name, f'A100 required, found {name}'; print(torch.ones(1, device='cuda'))"

In [ ]:
# 4. CONTROL PANEL — Sprint 4.5 GMU fold-0 screen (4 configs, fold 0).
import os
os.chdir('/content/hyperkvasir-multi-backbone-fusion')
os.environ['DRIVE_ROOT'] = '/content/drive/MyDrive/hyperkvasir-multi-cnn-fusion'
EXPERIMENTS = [
    'pair_vit_b_swin_t_gmu_cv',
    'pair_vit_b_beit_b_gmu_cv',
    'pair_swin_t_beit_b_gmu_cv',
    'triple_vit_swin_beit_gmu_cv',
]
FOLDS = [0]
SEEDS = [42]
# Stage 2 (only if a GMU config is competitive at fold 0): promote that single id
# to 5-fold by setting EXPERIMENTS = ['<best_gmu_cv>'] and FOLDS = [0, 1, 2, 3, 4].
n = len(EXPERIMENTS) * len(FOLDS) * len(SEEDS)
print('Drive root:', os.environ['DRIVE_ROOT'])
print(f'{len(EXPERIMENTS)} configs x {len(FOLDS)} folds x {len(SEEDS)} seeds = {n} runs')
print('Experiments:', *EXPERIMENTS, sep='\n  ')

In [ ]:
# 5. Stage the dataset to data/raw/hyperkvasir/labeled-images.
#    PREFERS a single archive on Drive (one sequential read — fast, avoids the
#    Drive FUSE small-file throttling that makes 10k per-file copies take hours).
#    Put labeled-images.zip (or .tar) next to the folder on Drive (see README
#    cell). Falls back to per-file copy if no archive exists. Verifies the staged
#    .jpg count against the fold manifest. Fine-tune reads images, not caches.
import os, csv, shutil
from pathlib import Path
os.chdir('/content/hyperkvasir-multi-backbone-fusion')
DRIVE = os.environ['DRIVE_ROOT']
dst = Path('data/raw/hyperkvasir/labeled-images')
with open('data/splits/hyperkvasir_official_5fold/fold_0.csv', newline='', encoding='utf-8-sig') as f:
    expected = sum(1 for _ in csv.DictReader(f))  # 10662 image rows
def count_images(p):
    # Tree also holds image-labels.csv + license.txt — count only .jpg/.jpeg.
    return sum(1 for _, _, fs in os.walk(p) for f in fs
               if f.lower().endswith(('.jpg', '.jpeg'))) if Path(p).exists() else 0
if dst.exists() and count_images(dst) == expected:
    print(f'[skip] already staged: {expected} images')
else:
    if dst.exists():
        print(f'[clean] partial copy {count_images(dst)}/{expected} -> removing', flush=True)
        shutil.rmtree(dst)
    dst.parent.mkdir(parents=True, exist_ok=True)
    archive = next((c for c in [
        f'{DRIVE}/data/hyperkvasir/labeled-images.zip',
        f'{DRIVE}/data/hyperkvasir/labeled-images.tar',
        f'{DRIVE}/data/labeled-images.zip',
    ] if os.path.exists(c)), None)
    if archive:
        local_arc = '/content/' + os.path.basename(archive)
        print(f'[archive] copying {archive} (one big file)...', flush=True)
        shutil.copy(archive, local_arc)
        print('[archive] extracting...', flush=True)
        shutil.unpack_archive(local_arc, 'data/raw/hyperkvasir')  # archive root = labeled-images/
    else:
        src = Path(f'{DRIVE}/data/hyperkvasir/labeled-images')
        if not src.is_dir():
            raise FileNotFoundError(f'No archive and no folder on Drive: {src}')
        files = sorted(p for p in src.rglob('*') if p.is_file())
        print(f'[per-file] no archive on Drive; copying {len(files)} files (SLOW; consider an archive)...', flush=True)
        for i, sp in enumerate(files, 1):
            tp = dst / sp.relative_to(src)
            tp.parent.mkdir(parents=True, exist_ok=True)
            shutil.copyfile(sp, tp)  # data only — fewer FUSE syscalls than copy2
            if i % 1000 == 0 or i == len(files):
                print(f'  {i}/{len(files)}', flush=True)
n = count_images(dst)
assert n == expected, f'staged {n} images != expected {expected} (archive must contain labeled-images/ at its root)'
print(f'[OK] staged {n} images -> {dst}')

In [ ]:
# 5b. (Sprint 3.5) A100 throughput benchmark — current vs fast.
#     Run AFTER cell 5 (dataset staged). Measurement only: it does NOT train,
#     touch configs, or write run artifacts. Skip cells 6-10 when benchmarking.
#     Needs scripts/benchmark_vit_throughput.py, which is on the Sprint3.5/perf
#     branch (cell 2 BRANCH already defaults to it).
import os
os.chdir('/content/hyperkvasir-multi-backbone-fusion')
!uv run --no-sync python scripts/benchmark_vit_throughput.py --experiment 02_single_swin_t_finetune_official --modes both --max-steps 100
!uv run --no-sync python scripts/benchmark_vit_throughput.py --experiment 11_triple_weighted_finetune_official --modes both --max-steps 60

In [ ]:
# 6. Dataset + git provenance gate (CNN D-09 gate; VLD-11). Idempotent.
#    Hashes the LOCAL staged tree (fast) against the manifest + git SHA; the Drive
#    archive is recorded as approved_source. The image pool is identical across
#    folds, so fold_0.csv is fine for the dataset-level provenance check.
import os, shutil, subprocess
from pathlib import Path
os.chdir('/content/hyperkvasir-multi-backbone-fusion')
prov = Path('results/vit/runs/sprint45_vit_gmu')
if prov.exists():
    shutil.rmtree(prov)
os.environ['EXPECTED_GIT_SHA'] = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
!uv run --no-sync python scripts/check_provenance.py --run-id sprint45_vit_gmu --source-dataset-root data/raw/hyperkvasir/labeled-images --staged-dataset-root data/raw/hyperkvasir/labeled-images --manifest data/splits/hyperkvasir_official_5fold/fold_0.csv --approved-source "$DRIVE_ROOT/data/hyperkvasir/labeled-images.tar" --expected-git-sha $EXPECTED_GIT_SHA --device cuda --output-root results/vit/runs

In [ ]:
# 7. Train the GMU configs (fold-0 screen) — overnight-safe.
#    Run-dir names match scripts/train.py: {id}[_seed{S}][_fold_{k}]. Restores
#    finished runs from Drive, skips runs with metrics.json, streams live
#    progress (python -u), and backs up each finished run to Drive immediately.
import os, shutil, subprocess
from pathlib import Path
os.chdir('/content/hyperkvasir-multi-backbone-fusion')
runs = Path('results/vit/runs')
backup = Path(os.environ['DRIVE_ROOT']) / 'returned_outputs' / 'sprint45_vit_gmu'
backup.mkdir(parents=True, exist_ok=True)

def run_name(exp, fold, seed):
    name = exp
    if seed != 42:
        name = f'{name}_seed{seed}'
    if fold != 0:
        name = f'{name}_fold_{fold}'
    return name

jobs = [(e, f, s) for e in EXPERIMENTS for f in FOLDS for s in SEEDS]

# Restore previously-finished runs from Drive (survives a fresh session).
for e, f, s in jobs:
    rn = run_name(e, f, s)
    if (backup / rn / 'metrics.json').exists() and not (runs / rn / 'metrics.json').exists():
        shutil.copytree(backup / rn, runs / rn, dirs_exist_ok=True)
        print(f'[restored] {rn}')

done = [(e, f, s) for e, f, s in jobs if (runs / run_name(e, f, s) / 'metrics.json').exists()]
todo = [j for j in jobs if j not in done]
print(f'done {len(done)}/{len(jobs)}; todo {len(todo)}\n', flush=True)

env = {**os.environ, 'PYTHONUNBUFFERED': '1'}  # stream child stdout live
for e, f, s in todo:
    rn = run_name(e, f, s)
    cmd = ['uv', 'run', '--no-sync', 'python', '-u', 'scripts/train.py',
           '--config', 'configs/vit/experiment_matrix.yaml',
           '--experiment', e, '--fold', str(f), '--device', 'cuda']
    if s != 42:
        cmd += ['--seed', str(s)]
    print(f'\n===== [run] {rn} =====', flush=True)
    subprocess.run(cmd, check=True, env=env)
    shutil.copytree(runs / rn, backup / rn, dirs_exist_ok=True)  # back up now
    print(f'[backed up] {rn}', flush=True)
print('\n[done] all requested runs finished')

In [ ]:
# 8. Artifact + finite-metric checklist for every (config, fold, seed) run.
import json, math
from pathlib import Path
runs = Path('results/vit/runs')
required = ['metrics.json', 'config.yaml', 'predictions.npz', 'best.pt']

def run_name(exp, fold, seed):
    name = exp
    if seed != 42:
        name = f'{name}_seed{seed}'
    if fold != 0:
        name = f'{name}_fold_{fold}'
    return name

jobs = [(e, f, s) for e in EXPERIMENTS for f in FOLDS for s in SEEDS]
problems = []
for e, f, s in jobs:
    rn = run_name(e, f, s)
    present = {r: (runs / rn / r).exists() for r in required}
    if all(present.values()):
        m = json.load((runs / rn / 'metrics.json').open())['test']
        finite = all(isinstance(m.get(k), (int, float)) and math.isfinite(m[k])
                     for k in ('macro_f1', 'accuracy', 'macro_precision', 'macro_recall'))
        print(f"{rn}: f1={m['macro_f1']:.4f} finite={finite}")
        if not finite:
            problems.append(f'{rn}: non-finite metric')
    else:
        print(f'{rn}: MISSING {present}')
        problems.append(f'{rn}: missing artifacts')
if problems:
    raise RuntimeError('Artifact/metric problems:\n  ' + '\n  '.join(problems))
print(f'\n[OK] all {len(jobs)} runs have the 4 artifacts and finite test metrics')

In [ ]:
# 9. Zip the GMU run dirs for direct download (also backed up per-run in cell 7).
import zipfile
from pathlib import Path
runs = Path('results/vit/runs')

def run_name(exp, fold, seed):
    name = exp
    if seed != 42:
        name = f'{name}_seed{seed}'
    if fold != 0:
        name = f'{name}_fold_{fold}'
    return name

zip_path = '/content/vit_gmu_runs.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for e in EXPERIMENTS:
        for f in FOLDS:
            for s in SEEDS:
                d = runs / run_name(e, f, s)
                for fp in d.rglob('*'):
                    if fp.is_file():
                        z.write(fp, fp.relative_to(runs))
print('[OK] zip:', zip_path)

In [ ]:
# 10. Download the zip to your machine (unzip into local results/vit/runs/).
from google.colab import files
files.download('/content/vit_gmu_runs.zip')